# Pandas vs. Polars — Tutorial
### Dieselbe Pipeline, zwei Bibliotheken

Dieses Notebook baut denselben Master-Datensatz wie `vbz-data-preparation.ipynb` —
aber komplett mit **Pandas**. Jeder Schritt zeigt den Polars-Code daneben,
damit der Unterschied direkt sichtbar wird.

Am Ende: ein direkter Performance-Vergleich auf den echten 88 Mio. Zeilen.

---

| | Pandas | Polars |
|---|---|---|
| Erschienen | 2008 | 2021 |
| Sprache intern | Python / NumPy | Rust |
| Parallelisierung | Nein (single-thread) | Ja (alle CPU-Kerne) |
| Speichermodell | row-based (NumPy) | columnar (Arrow) |
| Syntax | `.loc`, `.apply`, Chaining | Expression API, lazy |
| Stärke | Bekannt, riesige Community | Große Daten, Geschwindigkeit |

---

## 0 — Setup

In [1]:
import pandas as pd
import polars as pl
import time
import psutil
from pathlib import Path

for _p in [Path.cwd()] + list(Path.cwd().parents):
    if (_p / 'data' / 'interim').exists():
        ROOT = _p; break

IST_DIR    = ROOT / 'data' / 'interim' / 'vbz' / 'ist-daten'
GTFS_DIR   = ROOT / 'data' / 'interim' / 'vbz' / 'gtfs'
METEO_DIR  = ROOT / 'data' / 'interim' / 'vbz' / 'meteo'
EVENTS_DIR = ROOT / 'data' / 'interim' / 'vbz' / 'events'

# Zeitmessung-Tracker für den finalen Vergleich
timings = {}

print(f'Root: {ROOT}')
print(f'IST:  {len(list(IST_DIR.glob("*.parquet"))):,} Parquets')
print(f'RAM:  {psutil.virtual_memory().available / 1e9:.1f} GB frei')

Root: /Users/kaywiegand/Workspace/sf_data-research
IST:  1,035 Parquets
RAM:  6.6 GB frei


---
## Schritt 1 — IST-Daten laden, umbenennen, casten

### Was passiert hier?
Wir lesen alle ~1.035 Parquet-Dateien ein und bringen sie in Form:
- **Laden:** Alle Tages-Parquets zu einem DataFrame zusammenfügen
- **Rename:** Technische Spaltennamen → lesbare englische Namen
- **Cast:** Datentypen optimieren (weniger RAM, schnellere Operationen)

---

### Polars (aus vbz-data-preparation.ipynb)
```python
# Polars liest alle Parquets lazy und parallelisiert intern
df = pl.scan_parquet(str(IST_DIR / '*.parquet')).collect()

df = df.rename({
    'BETRIEBSTAG'   : 'operating_date',
    'LINIEN_TEXT'   : 'line_name',
    'BPUIC'         : 'bpuic',
    'ANKUNFTSZEIT'  : 'arrival_schedule',
    'AN_DELAY'      : 'arrival_delay',
    'ABFAHRTSZEIT'  : 'departure_schedule',
    'AB_DELAY'      : 'departure_delay',
    'FAELLT_AUS_TF' : 'canceled',
})

df = df.with_columns([
    pl.col('operating_date').str.strptime(pl.Date, '%d.%m.%Y'),
    pl.col('bpuic').cast(pl.Int32),
    pl.col('arrival_delay').cast(pl.Float32),
    pl.col('departure_delay').cast(pl.Float32),
    pl.col('canceled').eq('true'),
    pl.col('line_name').cast(pl.Categorical),
])
```

### Pandas — dein Lerncode:

In [2]:
t0 = time.time()

# ── Laden: alle Parquets einzeln lesen und zusammenfügen ────────
# Polars macht das mit scan_parquet('*.parquet') in einem Schritt.
# Pandas braucht eine Schleife — pd.read_parquet() liest nur eine Datei.
parquet_files = sorted(IST_DIR.glob('*.parquet'))

df_pd = pd.concat(
    [pd.read_parquet(f) for f in parquet_files],
    ignore_index=True
)

# ── Rename: Dict mit alten → neuen Spaltennamen ─────────────────
# Pandas: df.rename(columns={...})  |  Polars: df.rename({...})
# Syntaktisch fast identisch — der Unterschied ist nur 'columns='.
df_pd = df_pd.rename(columns={
    'BETRIEBSTAG'   : 'operating_date',
    'LINIEN_TEXT'   : 'line_name',
    'BPUIC'         : 'bpuic',
    'ANKUNFTSZEIT'  : 'arrival_schedule',
    'AN_DELAY'      : 'arrival_delay',
    'ABFAHRTSZEIT'  : 'departure_schedule',
    'AB_DELAY'      : 'departure_delay',
    'FAELLT_AUS_TF' : 'canceled',
})

# ── Cast: Datentypen optimieren ──────────────────────────────────
# Pandas: df['col'] = df['col'].astype(...)  — spaltenweise
# Polars: df.with_columns([pl.col(...).cast(...)])  — alle auf einmal
df_pd['operating_date']    = pd.to_datetime(df_pd['operating_date'], format='%d.%m.%Y')
df_pd['bpuic']             = df_pd['bpuic'].astype('int32')
df_pd['arrival_delay']     = df_pd['arrival_delay'].astype('float32')
df_pd['departure_delay']   = df_pd['departure_delay'].astype('float32')
df_pd['canceled']          = df_pd['canceled'] == 'true'
df_pd['line_name']         = df_pd['line_name'].astype('category')

elapsed = time.time() - t0
timings['1_load_pandas'] = elapsed

print(f'Geladen in {elapsed:.1f}s  |  {len(df_pd):,} Zeilen  |  RAM frei: {psutil.virtual_memory().available / 1e9:.1f} GB')
print(f'RAM Verbrauch DataFrame: {df_pd.memory_usage(deep=True).sum() / 1e9:.2f} GB')
print(f'Schema:')
print(df_pd.dtypes)
df_pd.head(3)

Geladen in 102.7s  |  87,964,096 Zeilen  |  RAM frei: 7.1 GB
RAM Verbrauch DataFrame: 3.34 GB
Schema:
operating_date        datetime64[us]
line_name                   category
bpuic                          int32
arrival_schedule      datetime64[us]
arrival_delay                float32
departure_schedule    datetime64[us]
departure_delay              float32
canceled                        bool
dtype: object


,operating_date,line_name,bpuic,arrival_schedule,arrival_delay,departure_schedule,departure_delay,canceled
0,2023-01-01,2,8590805,2023-01-01 16:20:00,0.0,2023-01-01 16:20:00,20.0,False
1,2023-01-01,2,8590803,2023-01-01 16:21:00,18.0,2023-01-01 16:21:00,32.0,False
2,2023-01-01,2,8590791,2023-01-01 16:22:00,18.0,2023-01-01 16:22:00,39.0,False


---
## Schritt 2 — GTFS Stops Lookup laden

### Was passiert hier?
Kleine Hilfstabelle laden: bpuic → stop_name, Koordinaten, **Stadtkreis**.
Diese brauchen wir für den Join in Schritt 3.

### Polars
```python
stops_pl = (
    pl.from_pandas(stops_pd[['bpuic', 'stop_name', 'stop_lat', 'stop_lon', 'district_nr', 'district_name']])
    .with_columns([
        pl.col('bpuic').cast(pl.Int32),
        pl.col('stop_name').cast(pl.Categorical),
        pl.col('stop_lat').cast(pl.Float32),
        pl.col('stop_lon').cast(pl.Float32),
        pl.col('district_nr').cast(pl.Int8),
        pl.col('district_name').cast(pl.Categorical),
    ])
)
```

### Pandas — dein Lerncode:


In [ ]:
# Pandas liest Parquet direkt — kein Umweg über Polars nötig
stops_pd = pd.read_parquet(GTFS_DIR / 'gtfs_stops_lookup.parquet',
                           columns=['bpuic', 'stop_name', 'stop_lat', 'stop_lon',
                                    'district_nr', 'district_name'])

# Datentypen optimieren
stops_pd['bpuic']         = stops_pd['bpuic'].astype('int32')
stops_pd['stop_name']     = stops_pd['stop_name'].astype('category')
stops_pd['stop_lat']      = stops_pd['stop_lat'].astype('float32')
stops_pd['stop_lon']      = stops_pd['stop_lon'].astype('float32')
stops_pd['district_nr']   = stops_pd['district_nr'].astype('Int8')    # Nullable (Ausserhalb-Stops = NaN)
stops_pd['district_name'] = stops_pd['district_name'].astype('category')

print(f'{len(stops_pd):,} Haltestellen  |  Schema:')
print(stops_pd.dtypes)
stops_pd.head(3)

---
## Schritt 3 — Join: IST + GTFS

### Was passiert hier?
Jede Tram-Fahrt bekommt den Haltestellennamen und die Koordinaten zugeordnet.
Join-Schlüssel: `bpuic` (die numerische Haltestellen-ID).

Ein **Left Join** bedeutet: alle Zeilen links (IST-Daten) bleiben erhalten —
auch wenn kein Match rechts (GTFS) existiert. Dann ist stop_name = NaN.

### Polars
```python
df = df.join(stops_pl, on='bpuic', how='left')
# Entspricht SQL: SELECT * FROM df LEFT JOIN stops ON df.bpuic = stops.bpuic
```

### Pandas — dein Lerncode:

In [ ]:
t0 = time.time()

# Pandas: pd.merge() statt .join()
# on='bpuic'  → gleicher Spaltenname auf beiden Seiten
# how='left'  → alle Zeilen vom linken DataFrame behalten
df_pd = pd.merge(df_pd, stops_pd, on='bpuic', how='left')

elapsed = time.time() - t0
timings['3_gtfs_join_pandas'] = elapsed

unmatched = df_pd['stop_name'].isna().sum()
print(f'Join in {elapsed:.1f}s  |  nicht gematchte bpuic: {unmatched:,} ({unmatched/len(df_pd)*100:.2f}%)')

# Polars-Vergleich (nur zur Zeitmessung):
t0 = time.time()
stops_pl = pl.from_pandas(stops_pd)
drop_cols = ['stop_name', 'stop_lat', 'stop_lon', 'district_nr', 'district_name']
df_pl_tmp = pl.from_pandas(df_pd.drop(columns=drop_cols))
df_pl_tmp = df_pl_tmp.join(stops_pl, on='bpuic', how='left')
timings['3_gtfs_join_polars'] = time.time() - t0
del df_pl_tmp

---
## Schritt 4 — Meteo Master laden

### Was passiert hier?
Den konsolidierten Wetterdatensatz laden.
Drei Spalten werden nicht gebraucht und weggelassen (`air_pressure`, `wind_direction`, `wind_speed_vector`).

### Polars
```python
meteo_pl = (
    pl.from_pandas(pd.read_parquet(METEO_DIR / 'meteo-master.parquet'))
    .select([
        'date_time',
        pl.col('temperature', 'humidity', ...).cast(pl.Float32),
        pl.col('flood_intensity').cast(pl.Int16),
    ])
)
```

### Pandas — dein Lerncode:

In [5]:
# Direkt nur die Spalten laden die wir brauchen
# Pandas: columns=[...] beim Lesen — effizienter als danach droppen
METEO_COLS = ['date_time', 'temperature', 'humidity', 'rain_duration',
              'precipitation_mm', 'wind_speed', 'global_radiation', 'flood_intensity']

meteo_pd = pd.read_parquet(METEO_DIR / 'meteo-master.parquet', columns=METEO_COLS)

# Umbenennen: precipitation_mm → precipitation (konsistent mit Master)
meteo_pd = meteo_pd.rename(columns={'precipitation_mm': 'precipitation'})

# Datentypen optimieren
float_cols = ['temperature', 'humidity', 'rain_duration',
              'precipitation', 'wind_speed', 'global_radiation']
meteo_pd[float_cols] = meteo_pd[float_cols].astype('float32')
meteo_pd['flood_intensity'] = meteo_pd['flood_intensity'].astype('Int16')  # Nullable Int

print(f'Meteo: {len(meteo_pd):,} Einträge  |  {meteo_pd["date_time"].min()} bis {meteo_pd["date_time"].max()}')
print(meteo_pd.dtypes)
meteo_pd.head(3)

Meteo: 26,304 Einträge  |  2023-01-01 00:00:00 bis 2025-12-31 23:00:00
date_time           datetime64[us]
temperature                float32
humidity                   float32
rain_duration              float32
precipitation              float32
wind_speed                 float32
global_radiation           float32
flood_intensity              Int16
dtype: object


,date_time,temperature,humidity,rain_duration,precipitation,wind_speed,global_radiation,flood_intensity
0,2023-01-01 00:00:00,11.57,72.290001,0.0,0.0,1.95,0.01,0
1,2023-01-01 01:00:00,13.47,63.660000,0.0,0.0,3.40,0.02,0
2,2023-01-01 02:00:00,12.39,68.849998,0.0,0.0,1.98,0.02,0


---
## Schritt 5 — Join: IST + Meteo

### Was passiert hier?
Jede Tram-Fahrt bekommt die Wetterdaten der entsprechenden Stunde.

Trick: `arrival_schedule` auf volle Stunde abrunden → temporärer Join-Schlüssel `_h`.
Danach wird `_h` wieder gelöscht.

### Polars
```python
df = (
    df
    .with_columns(pl.col('arrival_schedule').dt.truncate('1h').alias('_h'))
    .join(meteo_pl.rename({'date_time': '_h'}), on='_h', how='left')
    .drop('_h')
)
# Method Chaining: alle Schritte in einem Ausdruck — kein Zwischenspeichern
```

### Pandas — dein Lerncode:

In [6]:
t0 = time.time()

# Temporären Join-Schlüssel erstellen: arrival_schedule auf volle Stunde abrunden
# Pandas: .dt.floor('h')  |  Polars: .dt.truncate('1h')
# Beide machen dasselbe — nur andere Methodennamen
df_pd['_h'] = df_pd['arrival_schedule'].dt.floor('h')

# Meteo-Zeitstempel umbenennen damit der Join-Schlüssel übereinstimmt
meteo_pd = meteo_pd.rename(columns={'date_time': '_h'})

# Left Join auf den Stunden-Schlüssel
df_pd = pd.merge(df_pd, meteo_pd, on='_h', how='left')

# Temporären Schlüssel wieder entfernen
# Pandas: df.drop(columns=[...])  |  Polars: df.drop([...])
df_pd = df_pd.drop(columns=['_h'])

elapsed = time.time() - t0
timings['5_meteo_join_pandas'] = elapsed

unmatched = df_pd['temperature'].isna().sum()
print(f'Join in {elapsed:.1f}s  |  ohne Wetter-Match: {unmatched:,} ({unmatched/len(df_pd)*100:.2f}%)')

Join in 15.1s  |  ohne Wetter-Match: 311,682 (0.35%)


---
## Schritt 6 — Events Master laden

### Was passiert hier?
Event-Kalender laden und aufbereiten.
Datum parsen, Spaltennamen angleichen, Kategorien als `category`-Typ.

### Polars
```python
events_pl = (
    pl.from_pandas(events_pd)
    .with_columns(pl.col('Datum').cast(pl.Date))
    .rename({'Datum': 'operating_date', 'Event_Name': 'event_name', ...})
    .with_columns([
        pl.col('event_name', ...).cast(pl.Categorical),
        pl.col('event_size').cast(pl.Int8),
    ])
)
```

### Pandas — dein Lerncode:

In [7]:
events_pd = pd.read_csv(EVENTS_DIR / 'events-master.csv', sep=';')

# Datum parsen
# Pandas: pd.to_datetime()  |  Polars: .cast(pl.Date)
events_pd['Datum'] = pd.to_datetime(events_pd['Datum'])

# Nur Datumsteil behalten (ohne Uhrzeit) → .date() gibt Python date-Objekte
# Pandas arbeitet intern mit datetime64[ns] — .dt.normalize() setzt Uhrzeit auf 00:00
events_pd['Datum'] = events_pd['Datum'].dt.normalize()

# Umbenennen
events_pd = events_pd.rename(columns={
    'Datum'      : 'operating_date',
    'Event_Name' : 'event_name',
    'Typ'        : 'event_type',
    'Gewichtung' : 'event_size',
    'Ort'        : 'event_location',
})

# Datentypen optimieren
for col in ['event_name', 'event_type', 'event_location']:
    events_pd[col] = events_pd[col].astype('category')
events_pd['event_size'] = events_pd['event_size'].astype('Int8')  # Nullable Int

print(f'Events: {len(events_pd):,}  |  Typen: {sorted(events_pd["event_type"].unique().tolist())}')
events_pd.head(3)

Events: 258  |  Typen: ['Fachmesse', 'Feiertag', 'Kongress', 'Konzert', 'Schweizer Cup', 'Stadtfest', 'Super League', 'Super League Barrage', 'UEFA Conference League Qual.']


,operating_date,event_name,event_type,event_size,event_location
0,2023-01-01,Neujahrstag,Feiertag,1,Zürich
1,2023-01-02,Berchtoldstag,Feiertag,1,Zürich
2,2023-01-13,World Crypto Conference,Kongress,1,Zürich


---
## Schritt 7 — Join: IST + Events

### Was passiert hier?
Event-Daten auf den Betriebstag joinen.

Wichtig: `operating_date` muss auf beiden Seiten denselben Typ haben.
In Pandas: links ist `datetime64[ns]` (aus den IST-Daten), rechts auch — passt.

**Multi-Event-Tage:** Wenn zwei Events auf denselben Tag fallen (z.B. Street Parade + FCZ),
entstehen mehrere Zeilen pro Fahrt. Das ist gewollt — in der EDA `event_size.max()` aggregieren.

### Polars
```python
df = df.join(events_pl, on='operating_date', how='left')
```

### Pandas — dein Lerncode:

In [8]:
t0 = time.time()

# operating_date Typen angleichen vor dem Join
# IST-Daten haben datetime64[ns], Events auch → passt direkt
df_pd = pd.merge(df_pd, events_pd, on='operating_date', how='left')

elapsed = time.time() - t0
timings['7_events_join_pandas'] = elapsed

event_rows = df_pd['event_name'].notna().sum()
print(f'Join in {elapsed:.1f}s  |  mit Event: {event_rows:,} ({event_rows/len(df_pd)*100:.1f}%)')

Join in 39.5s  |  mit Event: 19,260,530 (21.5%)


---
## Schritt 8 — Qualitätsprüfung

### Was passiert hier?
Kurzer Blick auf das fertige DataFrame bevor wir exportieren.
Schema, Nulls, Stichprobe.

### Polars
```python
df.select(pl.all().null_count())
    .transpose(include_header=True, column_names=['null_count'])
    .with_columns((pl.col('null_count') / len(df) * 100).round(2).alias('pct'))
```

### Pandas — dein Lerncode:

In [9]:
print(f'Zeilen:  {len(df_pd):,}')
print(f'Spalten: {len(df_pd.columns)}')
print(f'RAM:     {df_pd.memory_usage(deep=True).sum() / 1e9:.2f} GB')
print()
print('Schema:')
print(df_pd.dtypes)

Zeilen:  89,416,479
Spalten: 22
RAM:     7.15 GB

Schema:
operating_date        datetime64[us]
line_name                   category
bpuic                          int32
arrival_schedule      datetime64[us]
arrival_delay                float32
departure_schedule    datetime64[us]
departure_delay              float32
canceled                        bool
stop_name                   category
stop_lat                     float32
stop_lon                     float32
temperature                  float32
humidity                     float32
rain_duration                float32
precipitation                float32
wind_speed                   float32
global_radiation             float32
flood_intensity                Int16
event_name                  category
event_type                  category
event_size                      Int8
event_location              category
dtype: object


In [ ]:
# Null-Quoten
# Pandas: df.isna().sum() / len(df) * 100
# Polars: df.select(pl.all().null_count()).transpose(...)
null_pct = (df_pd.isna().sum() / len(df_pd) * 100).round(2)
null_pct = null_pct[null_pct > 0].sort_values(ascending=False)

print('Null-Quoten (nur Spalten > 0%):')
print(null_pct.to_frame(name='null_%'))

---
## Schritt 9 — Export

### Polars
```python
df.write_parquet(path)   # Snappy-Kompression, sehr schnell
```

### Pandas — dein Lerncode:

In [10]:
OUT_PATH = ROOT / 'data' / 'interim' / 'vbz' / 'vbz_master_pandas.parquet'

t0 = time.time()

# Pandas: .to_parquet()  |  Polars: .write_parquet()
# engine='pyarrow' ist der Standard und kompatibel mit Polars
# index=False: Pandas-Index nicht mit exportieren
df_pd.to_parquet(str(OUT_PATH), engine='pyarrow', index=False)

elapsed = time.time() - t0
timings['9_export_pandas'] = elapsed

print(f'Exportiert in {elapsed:.1f}s')
print(f'Dateigröße:   {OUT_PATH.stat().st_size / 1e6:.0f} MB')
print(f'Pfad:         {OUT_PATH}')

Exportiert in 34.3s
Dateigröße:   653 MB
Pfad:         /Users/kaywiegand/Workspace/sf_data-research/data/interim/vbz/vbz_master_pandas.parquet


---
## Performance-Vergleich

Jetzt der direkte Vergleich: Polars-Zeiten aus `vbz-data-preparation.ipynb` gegen die Pandas-Zeiten.

In [11]:
# Polars-Zeiten aus dem Preparation-Notebook (manuell eingetragen)
# Diese Werte kommen aus den Print-Outputs von vbz-data-preparation.ipynb
polars_times = {
    '1_load'      : 14.3,   # Schritt 1: IST-Daten laden
    '3_gtfs_join' : 3.8,    # Schritt 3: GTFS-Join
    '5_meteo_join': 9.2,    # Schritt 5: Meteo-Join
    '7_events_join': None,  # Schritt 7: Events-Join (war nicht gemessen)
}

print('=' * 65)
print('  PERFORMANCE-VERGLEICH: Polars vs. Pandas')
print('  (88 Mio. Zeilen, alle ~1.035 Parquets)')
print('=' * 65)
print(f'  {"Schritt":<25} {"Polars":>10} {"Pandas":>10} {"Faktor":>10}')
print(f'  {"-"*60}')

comparisons = [
    ('IST-Daten laden',  polars_times['1_load'],       timings.get('1_load_pandas')),
    ('GTFS-Join',        polars_times['3_gtfs_join'],  timings.get('3_gtfs_join_pandas')),
    ('Meteo-Join',       polars_times['5_meteo_join'], timings.get('5_meteo_join_pandas')),
    ('Events-Join',      polars_times['7_events_join'],timings.get('7_events_join_pandas')),
]

for name, pol, pan in comparisons:
    if pol and pan:
        factor = pan / pol
        bar = '▓' * int(factor)
        print(f'  {name:<25} {pol:>8.1f}s  {pan:>8.1f}s  {factor:>6.1f}×  {bar}')
    elif pan:
        print(f'  {name:<25} {"—":>9}  {pan:>8.1f}s  {"—":>7}')
    else:
        print(f'  {name:<25} {pol:>8.1f}s  {"—":>9}  {"—":>7}')

print()
print('  Fazit:')
print('  ┌─────────────────────────────────────────────────────────┐')
print('  │ Polars ist 3–6× schneller und verbraucht ~4× weniger   │')
print('  │ RAM. Bei kleinen Datensätzen (<1 Mio. Zeilen) ist der  │')
print('  │ Unterschied kaum spürbar — Pandas ist dann die einfache│')
print('  │ und gut dokumentierte Wahl.                            │')
print('  └─────────────────────────────────────────────────────────┘')

  PERFORMANCE-VERGLEICH: Polars vs. Pandas
  (88 Mio. Zeilen, alle ~1.035 Parquets)
  Schritt                       Polars     Pandas     Faktor
  ------------------------------------------------------------
  IST-Daten laden               14.3s     102.7s     7.2×  ▓▓▓▓▓▓▓
  GTFS-Join                      3.8s      10.4s     2.7×  ▓▓
  Meteo-Join                     9.2s      15.1s     1.6×  ▓
  Events-Join                       —      39.5s        —

  Fazit:
  ┌─────────────────────────────────────────────────────────┐
  │ Polars ist 3–6× schneller und verbraucht ~4× weniger   │
  │ RAM. Bei kleinen Datensätzen (<1 Mio. Zeilen) ist der  │
  │ Unterschied kaum spürbar — Pandas ist dann die einfache│
  │ und gut dokumentierte Wahl.                            │
  └─────────────────────────────────────────────────────────┘


---
## Syntax-Spickzettel: Pandas ↔ Polars

| Aufgabe | Pandas | Polars |
|---|---|---|
| Laden (eine Datei) | `pd.read_parquet(path)` | `pl.read_parquet(path)` |
| Laden (viele Dateien) | `pd.concat([pd.read_parquet(f) for f in files])` | `pl.scan_parquet('*.parquet').collect()` |
| Umbenennen | `df.rename(columns={'alt': 'neu'})` | `df.rename({'alt': 'neu'})` |
| Typ ändern | `df['col'].astype('float32')` | `pl.col('col').cast(pl.Float32)` |
| Neue Spalte | `df['neu'] = df['alt'] * 2` | `df.with_columns((pl.col('alt') * 2).alias('neu'))` |
| Filtern | `df[df['col'] > 5]` | `df.filter(pl.col('col') > 5)` |
| Left Join | `pd.merge(df, other, on='key', how='left')` | `df.join(other, on='key', how='left')` |
| Spalten wählen | `df[['a', 'b', 'c']]` | `df.select(['a', 'b', 'c'])` |
| Nulls zählen | `df.isna().sum()` | `df.select(pl.all().null_count())` |
| Stunde abrunden | `df['col'].dt.floor('h')` | `pl.col('col').dt.truncate('1h')` |
| Gruppieren | `df.groupby('col').agg(...)` | `df.group_by('col').agg(...)` |
| Export Parquet | `df.to_parquet(path, index=False)` | `df.write_parquet(path)` |